In [1]:
import sys
sys.path.append('..')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import src.db as db
from sklearn.model_selection import train_test_split

In [3]:
engine = db.get_engine()
with open("../sql/06_training_features.sql", encoding="utf-8") as f:
    query = f.read()

training_features = pd.read_sql_query(query, engine)

DATABASE_URL is set.


In [4]:
training_features.head()

,seller_id,activated,won_date,first_contact_date,business_type,lead_type,origin,days_to_convert
0,2aeedd049dc20e7171f43110fd0d6821,not activated,2018-06-13 16:37:42,2018-04-06,reseller,industry,unknown,68
1,a56e351445e5863fc8960640ba9190c7,activated,2018-02-26 20:02:10,2018-02-21,reseller,online_small,direct_traffic,5
2,76bf0e3e7d311d9069d4512cc1a232d7,not activated,2018-04-24 14:19:33,2018-04-03,manufacturer,online_small,unknown,21
3,b6a59a960c14aab3c683a673d03ca2ab,not activated,2018-01-23 11:33:43,2017-11-07,reseller,offline,social,77
4,1a932caad4f9d804097d7f8e615baed1,not activated,2018-02-19 19:51:00,2018-02-15,reseller,industry,unknown,4


In [5]:
training_features.shape

(707, 8)

In [6]:
training_features.dtypes

seller_id               str
activated               str
won_date                str
first_contact_date      str
business_type           str
lead_type               str
origin                  str
days_to_convert       int64
dtype: object

In [7]:
training_features.head()

,seller_id,activated,won_date,first_contact_date,business_type,lead_type,origin,days_to_convert
0,2aeedd049dc20e7171f43110fd0d6821,not activated,2018-06-13 16:37:42,2018-04-06,reseller,industry,unknown,68
1,a56e351445e5863fc8960640ba9190c7,activated,2018-02-26 20:02:10,2018-02-21,reseller,online_small,direct_traffic,5
2,76bf0e3e7d311d9069d4512cc1a232d7,not activated,2018-04-24 14:19:33,2018-04-03,manufacturer,online_small,unknown,21
3,b6a59a960c14aab3c683a673d03ca2ab,not activated,2018-01-23 11:33:43,2017-11-07,reseller,offline,social,77
4,1a932caad4f9d804097d7f8e615baed1,not activated,2018-02-19 19:51:00,2018-02-15,reseller,industry,unknown,4


In [8]:
training_features.drop(["seller_id", "won_date", "first_contact_date"], axis=1, inplace=True)

In [9]:
training_features.head()

,activated,business_type,lead_type,origin,days_to_convert
0,not activated,reseller,industry,unknown,68
1,activated,reseller,online_small,direct_traffic,5
2,not activated,manufacturer,online_small,unknown,21
3,not activated,reseller,offline,social,77
4,not activated,reseller,industry,unknown,4


In [10]:
training_features['activated'].value_counts()

activated
not activated    464
activated        243
Name: count, dtype: int64

In [11]:
training_features["activated"] = training_features["activated"].map({"activated": 1, "not activated": 0})

In [12]:
training_features.head()

,activated,business_type,lead_type,origin,days_to_convert
0,0,reseller,industry,unknown,68
1,1,reseller,online_small,direct_traffic,5
2,0,manufacturer,online_small,unknown,21
3,0,reseller,offline,social,77
4,0,reseller,industry,unknown,4


In [13]:
training_features.shape

(707, 5)

In [14]:
X = training_features.drop("activated", axis=1)
y = training_features["activated"]

In [15]:
y.shape

(707,)

In [16]:
X.shape

(707, 4)

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [18]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((565, 4), (142, 4), (565,), (142,))

In [19]:
y.value_counts().sum()



np.int64(707)

In [20]:
print(y.value_counts()[0]/y.value_counts().sum() * 100)
print(y.value_counts()[1]/y.value_counts().sum() * 100)


65.62942008486563
34.37057991513437


In [21]:
print(y_train.value_counts()[0]/y_train.value_counts().sum() * 100)
print(y_train.value_counts()[1]/y_train.value_counts().sum() * 100)

65.66371681415929
34.33628318584071


In [22]:
print(y_test.value_counts()[0]/y_test.value_counts().sum() * 100)
print(y_test.value_counts()[1]/y_test.value_counts().sum() * 100)

65.49295774647888
34.50704225352113


In [23]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [24]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
dummy_predictions = dummy_clf.predict(X_test)
score = accuracy_score(y_test, dummy_predictions)
print(f"Dummy Classifier Accuracy: {score * 100:.2f}%")

Dummy Classifier Accuracy: 65.49%


In [25]:
pd.Series(dummy_predictions).value_counts()

0    142
Name: count, dtype: int64

In [26]:
confusion_matrix(y_test, dummy_predictions)

array([[93,  0],
       [49,  0]])

In [27]:
classification_report(y_test, dummy_predictions)

c:\Users\nannu\cac-ltv-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nannu\cac-ltv-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nannu\cac-ltv-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape

'              precision    recall  f1-score   support\n\n           0       0.65      1.00      0.79        93\n           1       0.00      0.00      0.00        49\n\n    accuracy                           0.65       142\n   macro avg       0.33      0.50      0.40       142\nweighted avg       0.43      0.65      0.52       142\n'

In [28]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
categorical_features

C:\Users\nannu\AppData\Local\Temp\ipykernel_18128\3636180073.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=['object']).columns.tolist()


['business_type', 'lead_type', 'origin']

In [29]:
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_features

['days_to_convert']

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ct = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)

In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

clf = Pipeline(steps=[
    ("preprocessor", ct),
    ("classifier", LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](4,)","['business_type','lead_type','origin','days_to_convert']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining co

In [32]:
logistic_predictions = clf.predict(X_test)

In [33]:
pd.Series(logistic_predictions).value_counts()

0    115
1     27
Name: count, dtype: int64

In [34]:
clf.predict_proba(X_test)[:, 1]

array([0.41023522, 0.43167542, 0.41185273, 0.11861792, 0.42632098,
       0.25038107, 0.71516206, 0.29392507, 0.29515469, 0.28357986,
       0.35035112, 0.11908272, 0.36160997, 0.26297774, 0.3857994 ,
       0.46083272, 0.53016919, 0.33686583, 0.360245  , 0.46340686,
       0.26483314, 0.20490423, 0.51827097, 0.26585522, 0.25213079,
       0.47150908, 0.54360024, 0.4212007 , 0.38164397, 0.39237781,
       0.52317895, 0.46598295, 0.41238454, 0.44987857, 0.30256371,
       0.17859192, 0.54102979, 0.37982928, 0.47962634, 0.51209749,
       0.34913579, 0.360245  , 0.41059321, 0.15773807, 0.41838774,
       0.07282133, 0.32963418, 0.24294489, 0.47335274, 0.30171924,
       0.54401407, 0.29660635, 0.42741306, 0.24376212, 0.51653211,
       0.52576175, 0.44548889, 0.33675443, 0.32857424, 0.47445929,
       0.25315372, 0.25288844, 0.21552395, 0.39935161, 0.53992749,
       0.17371694, 0.26154654, 0.42415145, 0.53348441, 0.5406624 ,
       0.37733588, 0.54139713, 0.12048658, 0.36297718, 0.54543

In [35]:
confusion_matrix(y_test, logistic_predictions)

array([[75, 18],
       [40,  9]])

In [36]:
from sklearn.metrics import accuracy_score, precision_score , average_precision_score, recall_score, f1_score, roc_auc_score

In [37]:
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
logistic_precision = precision_score(y_test, logistic_predictions)
logistic_recall = recall_score(y_test, logistic_predictions)
logistic_f1 = f1_score(y_test, logistic_predictions)
logistic_roc_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])
logistic_average_precision = average_precision_score(y_test, clf.predict_proba(X_test)[:, 1])

In [38]:
print(f"Accuracy: {logistic_accuracy:.3f}")
print(f"Precision: {logistic_precision:.3f}")
print(f"Recall: {logistic_recall:.3f}")
print(f"F1 score: {logistic_f1:.3f}")
print(f"ROC-AUC: {logistic_roc_auc:.3f}")
print(f"Average precision: {logistic_average_precision:.3f}")

Accuracy: 0.592
Precision: 0.333
Recall: 0.184
F1 score: 0.237
ROC-AUC: 0.563
Average precision: 0.385


In [39]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [40]:
random_forest_clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])



In [41]:
gradient_boosting_clf = Pipeline(steps=[
    ("preprocessor", preprocessor), 
    ("classifier", GradientBoostingClassifier(random_state=42))
])

ada_boost_clf = Pipeline(steps=[
    ("preprocessor", preprocessor), 
    ("classifier", AdaBoostClassifier(random_state=42))
])

xgb_clf = Pipeline(steps=[
    ("preprocessor", preprocessor), 
    ("classifier", XGBClassifier(objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    n_jobs=-1))
])

In [42]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "average_precision"
]



In [43]:
random_forest_results = cross_validate(random_forest_clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

In [44]:
random_forest_results_df = pd.DataFrame(random_forest_results).mean().to_frame().T
random_forest_results_df

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_average_precision
0,0.636252,0.160047,0.578761,0.390759,0.355466,0.36938,0.528581,0.394542


In [46]:
print("Mean Recall:", random_forest_results["test_recall"].mean())
print("Std Recall:", random_forest_results["test_recall"].std())

Mean Recall: 0.35546558704453435
Std Recall: 0.02748004185956369


In [48]:
gradient_boodt_results = cross_validate(gradient_boosting_clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

gradient_boost_results_df = pd.DataFrame(gradient_boodt_results).mean().to_frame().T
gradient_boost_results_df



,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_average_precision
0,0.274429,0.05835,0.612389,0.353386,0.180297,0.234619,0.572124,0.396349


In [49]:
print("Mean Recall:", gradient_boodt_results["test_recall"].mean())
print("Std Recall:", gradient_boodt_results["test_recall"].std())

Mean Recall: 0.18029689608636973
Std Recall: 0.08710717598703181


In [50]:
ada_boost_results = cross_validate(ada_boost_clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

ada_boost_results_df = pd.DataFrame(ada_boost_results).mean().to_frame().T
ada_boost_results_df

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_average_precision
0,0.216304,0.092654,0.640708,0.449549,0.195682,0.269112,0.634027,0.476224


In [51]:
print("Mean Recall:", ada_boost_results["test_recall"].mean())
print("Std Recall:", ada_boost_results["test_recall"].std())

Mean Recall: 0.19568151147098517
Std Recall: 0.07134905672021409


In [52]:
xgb_results = cross_validate(xgb_clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

xgb_results_df = pd.DataFrame(xgb_results).mean().to_frame().T
xgb_results_df

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_average_precision
0,0.106522,0.05672,0.60708,0.346537,0.175169,0.227327,0.585747,0.410036


In [53]:
print("Mean Recall:", xgb_results["test_recall"].mean())
print("Std Recall:", xgb_results["test_recall"].std())

Mean Recall: 0.17516869095816462
Std Recall: 0.0815427454280163


In [57]:
logistic_reg_clf = Pipeline(steps=[
    ("preprocessor", ct),     
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

In [58]:
logistic_results = cross_validate(logistic_reg_clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

logistic_results_df = pd.DataFrame(logistic_results).mean().to_frame().T
logistic_results_df

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_average_precision
0,0.046961,0.058298,0.661947,0.506169,0.196086,0.276347,0.649083,0.493722


In [59]:
print("Mean Recall:", logistic_results["test_recall"].mean())
print("Std Recall:", logistic_results["test_recall"].std())

Mean Recall: 0.19608636977058028
Std Recall: 0.08574079344676391
